# DriveSense-VLM — 00: Data Pipeline

**GPU**: T4 (CPU/data-bound) | **Time**: ~2–3 h | **Cost**: ~5 CU

Runs the complete data curation pipeline:
1. nuScenes mini download + rarity filtering (6-signal composite score)
2. PySpark distributed ETL + analytics
3. DADA-2000 critical-moment extraction (optional, 53 GB)
4. Unified dataset assembly with stratified train/val/test splits
5. LLM counterfactual annotation (mock or real Anthropic API)
6. SFT JSONL formatting for Qwen3-VL training

> **RECOVERY**: If Colab disconnects, rerun Cells 2–3 (setup + install), then
> continue from the interrupted section. Every step skips if its output exists.

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Install data pipeline dependencies
# IMPORTANT: Do NOT install numpy or pandas — Colab's pre-installed versions
# are compiled against numpy 2.x ABI. Installing numpy<2 breaks pandas/opencv.
# nuscenes-devkit installed with --no-deps to skip its strict numpy version pin.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install nuscenes-devkit --no-deps -q 2>&1 | tail -1
!pip install pyquaternion matplotlib tqdm Pillow pyyaml requests openpyxl -q 2>&1 | tail -1
!pip install scikit-learn scipy pyspark -q 2>&1 | tail -1

# Verify imports
import numpy, pandas, drivesense
print(f"✓ numpy   {numpy.__version__}")
print(f"✓ pandas  {pandas.__version__}")
print(f"✓ drivesense importable from {drivesense.__file__}")
from nuscenes.nuscenes import NuScenes
print(f"✓ nuscenes-devkit importable")

## 1. nuScenes Mini Download

nuScenes mini is ~1 GB and sufficient for a full pipeline test.

**Option A** (automatic): The cell below tries `wget` from the official URL.
**Option B** (manual): Upload `v1.0-mini.tgz` to your Drive and run the fallback cell.

nuScenes registration: https://www.nuscenes.org/nuscenes#download

In [ ]:
import os

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
MINI_DIR     = f"{NUSCENES_DIR}/v1.0-mini"
SAMPLES_DIR  = f"{NUSCENES_DIR}/samples"

if os.path.exists(MINI_DIR) and os.path.exists(SAMPLES_DIR):
    print(f"✓ nuScenes mini already extracted — skipping download.")
else:
    os.makedirs(NUSCENES_DIR, exist_ok=True)
    print("Downloading nuScenes mini (~1 GB)...")
    !wget -q "https://www.nuscenes.org/data/v1.0-mini.tgz" \
          -O /tmp/nuscenes_mini.tgz || true

    tgz = "/tmp/nuscenes_mini.tgz"
    if os.path.exists(tgz) and os.path.getsize(tgz) > 1_000_000:
        print("Extracting...")
        !tar -xzf {tgz} -C {NUSCENES_DIR}
        !rm {tgz}
        print(f"✓ Extracted to {NUSCENES_DIR}")
    else:
        print("⚠ Auto-download failed (URL may require authentication).")
        print("  Upload v1.0-mini.tgz to Drive and run the fallback cell below.")

In [ ]:
# FALLBACK: Extract from manually uploaded archive on Drive
import os, glob

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
already_done = (os.path.exists(f"{NUSCENES_DIR}/v1.0-mini") and
                os.path.exists(f"{NUSCENES_DIR}/samples"))

if already_done:
    print("✓ nuScenes already extracted — skipping.")
else:
    # Search Drive for any .tgz with 'nuscenes' or 'mini' in name
    search_paths = [PROJECT_ROOT, "/content/drive/MyDrive"]
    candidates = []
    for base in search_paths:
        candidates += glob.glob(f"{base}/**/*.tgz", recursive=True)
    tgz = next((c for c in candidates
                if any(k in c.lower() for k in ("nuscenes", "mini", "v1.0"))), None)

    if tgz:
        print(f"Found: {tgz}")
        os.makedirs(NUSCENES_DIR, exist_ok=True)
        !tar -xzf {tgz} -C {NUSCENES_DIR}
        print(f"✓ Extracted to {NUSCENES_DIR}")
    else:
        print(f"⚠ No archive found. Upload v1.0-mini.tgz to {PROJECT_ROOT}/ then rerun.")

In [ ]:
# Verify nuScenes loads correctly
import os
from nuscenes.nuscenes import NuScenes

NUSCENES_DIR = f"{DATA_ROOT}/nuscenes"
print(f"Contents of {NUSCENES_DIR}:")
!ls {NUSCENES_DIR} 2>/dev/null || echo "  (directory empty or missing)"

if os.path.exists(f"{NUSCENES_DIR}/v1.0-mini"):
    try:
        nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_DIR, verbose=False)
        print(f"\n✓ nuScenes v1.0-mini loaded successfully:")
        print(f"  Scenes      : {len(nusc.scene)}")
        print(f"  Samples     : {len(nusc.sample)}")
        print(f"  Annotations : {len(nusc.sample_annotation)}")
    except Exception as e:
        print(f"\n✗ Load failed: {e}")
        print(f"  Expected layout inside {NUSCENES_DIR}/:")
        print(f"    v1.0-mini/   samples/   maps/")
else:
    print("\n⚠ v1.0-mini/ not found — complete the download step first.")

## 2. Rarity Filtering

Scores each frame on 6 binary signals (proximity, occlusion, density, adverse weather,
VRU at intersection, cyclist). Keeps frames with composite score ≥ threshold.
Use `--min-score 1` for the mini dataset to retain enough frames.

In [ ]:
import os, json

FILTER_OUTPUT = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
NUSCENES_DIR  = f"{DATA_ROOT}/nuscenes"
os.makedirs(FILTER_OUTPUT, exist_ok=True)

if os.path.exists(f"{FILTER_OUTPUT}/metadata.jsonl"):
    with open(f"{FILTER_OUTPUT}/metadata.jsonl") as f:
        n = sum(1 for _ in f)
    print(f"✓ Already filtered ({n} frames) — skipping.")
else:
    print("Running nuScenes rarity filter...")
    !python scripts/run_nuscenes_filter.py \
        --nuscenes-root {NUSCENES_DIR} \
        --version v1.0-mini \
        --output-dir {FILTER_OUTPUT} \
        --min-score 1 \
        2>&1

    # ── CRITICAL: Convert JSON array → JSONL ──────────────────────────────
    # run_nuscenes_filter.py writes metadata.json (array).
    # run_build_unified_dataset.py expects metadata.jsonl (one object per line).
    json_path  = f"{FILTER_OUTPUT}/metadata.json"
    jsonl_path = f"{FILTER_OUTPUT}/metadata.jsonl"

    if os.path.exists(json_path) and not os.path.exists(jsonl_path):
        with open(json_path) as f:
            frames = json.load(f)
        with open(jsonl_path, 'w') as f:
            for frame in frames:
                f.write(json.dumps(frame) + '\n')
        print(f"  ✓ Converted {len(frames)} frames → metadata.jsonl")
    elif os.path.exists(jsonl_path):
        with open(jsonl_path) as f:
            n = sum(1 for _ in f)
        print(f"  ✓ metadata.jsonl already exists ({n} frames)")
    else:
        print("  ⚠ metadata.json not produced — check filter output above")

!ls -lh {FILTER_OUTPUT} 2>/dev/null

## 3. PySpark ETL

Distributed rarity scoring with signal co-occurrence analytics and temporal clustering.
Runs in local Spark mode — no cluster needed.

In [ ]:
import os

SPARK_OUTPUT = f"{OUTPUTS_ROOT}/data/spark_processed"

if os.path.exists(SPARK_OUTPUT) and os.listdir(SPARK_OUTPUT):
    print(f"✓ Spark output already exists — skipping.")
    !ls {SPARK_OUTPUT}
else:
    os.makedirs(SPARK_OUTPUT, exist_ok=True)
    # JAVA_HOME required for PySpark on Colab
    os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-11-openjdk-amd64')
    print("Running PySpark rarity scoring pipeline...")
    !python scripts/run_spark_pipeline.py \
        --version v1.0-mini \
        --nuscenes-root {DATA_ROOT}/nuscenes \
        --output-dir {SPARK_OUTPUT} \
        2>&1
    print("✓ Spark pipeline complete.")
    !ls {SPARK_OUTPUT}

## 4. DADA-2000 (Optional)

DADA-2000 is a 53 GB accident dashcam dataset. **Skip for a mini/demo run.**

To include it:
1. Download from https://github.com/JWFangit/LOTVS-DADA
2. Upload to Drive: `DriveSense-VLM/data/dada2000/DADA-2000/`
3. Structure: `DADA-2000/<category>/<sequence>/images/`
4. Also upload `dada_text_annotations.xlsx` to `data/dada2000/`

The cell below auto-detects the data and skips gracefully if absent.

In [ ]:
import os

DADA_ROOT   = f"{DATA_ROOT}/dada2000"
DADA_DATA   = f"{DADA_ROOT}/DADA-2000"
DADA_OUTPUT = f"{OUTPUTS_ROOT}/data/dada_extracted"

if os.path.exists(DADA_DATA) and os.listdir(DADA_DATA):
    print(f"✓ DADA-2000 found at {DADA_DATA}")
    if os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl"):
        print(f"✓ Already extracted — skipping.")
    else:
        os.makedirs(DADA_OUTPUT, exist_ok=True)
        print("Running DADA-2000 extraction...")
        !python scripts/run_dada_extraction.py \
            --dada-root {DADA_ROOT} \
            --output-dir {DADA_OUTPUT} \
            2>&1
        print("✓ DADA-2000 extraction complete.")
        !ls {DADA_OUTPUT}
else:
    print("⚠ DADA-2000 not found — skipping (mini run mode).")
    print(f"  To enable: upload data to {DADA_DATA}/")

## 5. Unified Dataset

Merges nuScenes filtered frames + DADA-2000 (if present) into stratified
train/val/test splits using `StratifiedShuffleSplit` on source × category.

In [ ]:
import os, json

FILTER_OUTPUT  = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
DADA_OUTPUT    = f"{OUTPUTS_ROOT}/data/dada_extracted"
UNIFIED_OUTPUT = f"{OUTPUTS_ROOT}/data/unified"
os.makedirs(UNIFIED_OUTPUT, exist_ok=True)

if os.path.exists(f"{UNIFIED_OUTPUT}/manifest.json"):
    print(f"✓ Unified dataset already built — skipping.")
    !ls {UNIFIED_OUTPUT}
else:
    HAS_DADA  = os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl")
    dada_flag = f"--dada-dir {DADA_OUTPUT}" if HAS_DADA else "--nuscenes-only"
    print(f"Building unified dataset ({'nuScenes + DADA-2000' if HAS_DADA else 'nuScenes only'})...")

    !python scripts/run_build_unified_dataset.py \
        --nuscenes-dir {FILTER_OUTPUT} \
        --output-dir {UNIFIED_OUTPUT} \
        {dada_flag} \
        2>&1

    if os.path.exists(f"{UNIFIED_OUTPUT}/manifest.json"):
        with open(f"{UNIFIED_OUTPUT}/manifest.json") as f:
            manifest = json.load(f)
        print("\n✓ Unified dataset built:")
        for split, items in manifest.items():
            if isinstance(items, list):
                print(f"  {split}: {len(items)} frames")
    else:
        print("⚠ manifest.json not produced — check output above")

## 6. LLM Annotation

Generates structured JSON annotations (bbox, hazard class, severity, reasoning, action).
~30% of frames with hazards receive LLM counterfactual augmentation.

- **Mock mode** (no API key): placeholder annotations for pipeline testing.
- **Real mode**: set `ANTHROPIC_API_KEY` below — costs ~$5–10 for 800–1200 frames.

Per-frame cache enables safe resume if the session disconnects mid-annotation.

In [ ]:
import os

UNIFIED_OUTPUT = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATED_DIR  = f"{OUTPUTS_ROOT}/data/annotated"
CACHE_DIR      = f"{OUTPUTS_ROOT}/data/annotation_cache"
os.makedirs(ANNOTATED_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

if os.path.exists(f"{ANNOTATED_DIR}/annotated_manifest.json"):
    print("✓ Annotation already complete — skipping.")
else:
    # Uncomment for real LLM annotation (~$5-10, needs Anthropic key):
    # import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

    USE_MOCK  = 'ANTHROPIC_API_KEY' not in os.environ
    mock_flag = "--mock-llm" if USE_MOCK else ""
    print(f"Running annotation pipeline (mock={USE_MOCK})...")

    !python scripts/run_annotation_pipeline.py \
        --unified-dir {UNIFIED_OUTPUT} \
        --output-dir {ANNOTATED_DIR} \
        --cache-dir {CACHE_DIR} \
        {mock_flag} \
        2>&1

    if os.path.exists(f"{ANNOTATED_DIR}/annotated_manifest.json"):
        print(f"✓ Annotation complete → {ANNOTATED_DIR}/annotated_manifest.json")
    else:
        print("⚠ annotated_manifest.json not found — check output above")

## 7. SFT Formatting

Formats annotations into Qwen3-VL chat-format JSONL:
`[system, user(image+text), assistant(JSON)]` per example.

In [ ]:
import os, json

ANNOTATED_DIR = f"{OUTPUTS_ROOT}/data/annotated"
SFT_DIR       = f"{OUTPUTS_ROOT}/data/sft_ready"
os.makedirs(SFT_DIR, exist_ok=True)

if os.path.exists(f"{SFT_DIR}/sft_train.jsonl"):
    print("✓ SFT files already created — skipping.")
else:
    print("Formatting annotations into SFT JSONL...")
    !python scripts/run_annotation_pipeline.py \
        --format-only \
        --annotated-dir {ANNOTATED_DIR} \
        --output-dir {SFT_DIR} \
        2>&1

print("\nSFT Dataset Statistics:")
print("─" * 36)
total = 0
for split in ("train", "val", "test"):
    p = f"{SFT_DIR}/sft_{split}.jsonl"
    if os.path.exists(p):
        with open(p) as f:
            n = sum(1 for _ in f)
        total += n
        size_kb = os.path.getsize(p) / 1024
        print(f"  {split:<6}: {n:>5} examples  ({size_kb:.0f} KB)")
    else:
        print(f"  {split:<6}: NOT FOUND")
print(f"  {'Total':<6}: {total:>5} examples")

## 8. Verification

Final check of all pipeline outputs.

In [ ]:
import os, json

SFT_DIR      = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_OUT   = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
UNIFIED_OUT  = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATED    = f"{OUTPUTS_ROOT}/data/annotated"

checks = [
    ("nuScenes metadata.jsonl",  f"{FILTER_OUT}/metadata.jsonl"),
    ("Unified manifest.json",    f"{UNIFIED_OUT}/manifest.json"),
    ("Annotated manifest",       f"{ANNOTATED}/annotated_manifest.json"),
    ("sft_train.jsonl",          f"{SFT_DIR}/sft_train.jsonl"),
    ("sft_val.jsonl",            f"{SFT_DIR}/sft_val.jsonl"),
    ("sft_test.jsonl",           f"{SFT_DIR}/sft_test.jsonl"),
]

print("=" * 52)
print("  Pipeline Verification")
print("=" * 52)
all_ok = True
for label, path in checks:
    ok = os.path.exists(path)
    print(f"  {'✓' if ok else '✗'}  {label}")
    all_ok = all_ok and ok
print("=" * 52)

if all_ok:
    print("\n✓ All outputs present — ready for Notebook 01 (Training)")
    with open(f"{SFT_DIR}/sft_train.jsonl") as f:
        ex = json.loads(f.readline())
    print(f"\nSample keys   : {list(ex.keys())}")
    if "messages" in ex:
        print(f"Message roles : {[m.get('role') for m in ex['messages']]}")
else:
    print("\n⚠ Some outputs missing — rerun the failed cells above.")
    print("  Each step is idempotent (safe to rerun).")